# Retail Case Study Project

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DecimalType, DateType

In [0]:
# current Spark version and catalog
print(f"Spark version: {spark.version}")
print(f"Current catalog: {spark.catalog.currentCatalog()}")

## Bronze Layer

In [0]:
VOLUME_PATH = "/Volumes/workspace/retail_fresher/retail_raw"
raw_customers = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/customers.csv")
raw_products = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/products.csv")
raw_sales_orders = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/sales_orders.csv")

In [0]:
bronze_customers = (
    raw_customers
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/customers.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_products = (
    raw_products
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/products.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_sales_orders = (
    raw_sales_orders
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/sales_orders.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_customers")
bronze_products.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_products")
bronze_sales_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_sales_orders")

## PySpark Transformations

### Step 1

In [0]:
customers = spark.read.table("workspace.retail_fresher.bronze_customers")
print("Bronze customer rows", customers.count())
customers.printSchema()

In [0]:
products = spark.read.table("workspace.retail_fresher.bronze_products")
print("Bronze product rows", products.count())
products.printSchema()

In [0]:
sales_orders = spark.read.table("workspace.retail_fresher.bronze_sales_orders")
print("Bronze sales order rows", sales_orders.count())
sales_orders.printSchema()

### Steps 2 & 3

In [0]:
# customers
customers = (
    customers
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn(
        "city",
        F.when(F.col("city").isNull() | (F.col("city") == ""), "Unknown").otherwise(F.col("city"))
    )
    .withColumn("state", F.initcap(F.trim(F.col("state"))))
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
    .withColumn("customer_segment", F.initcap(F.trim(F.col("customer_segment"))))
    .withColumn("is_active", F.upper(F.trim(F.col("is_active"))))
)

In [0]:
# products
products = (
    products
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("subcategory", F.initcap(F.trim(F.col("subcategory"))))
    .withColumn("supplier_name", F.initcap(F.trim(F.col("supplier_name"))))
    .withColumn("active_flag", F.upper(F.trim(F.col("active_flag"))))
)

In [0]:
# sales orders
sales_orders = (
    sales_orders
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("payment_method", F.upper(F.trim(F.col("payment_method"))))
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("sales_channel", F.upper(F.trim(F.col("sales_channel"))))
    .withColumn("warehouse_id", F.trim(F.col("warehouse_id")))
)

### Step 4

In [0]:
# customers
customers = (
    customers
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("updated_at", F.to_timestamp(F.col("updated_at"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("loyalty_points", F.col("loyalty_points").try_cast(IntegerType()))
)

In [0]:
# products
products = (
    products
    .withColumn("unit_price", F.col("unit_price").try_cast(DecimalType(10, 2)))
    .withColumn("cost_price", F.col("cost_price").try_cast(DecimalType(10, 2)))
    .withColumn("stock_quantity", F.col("stock_quantity").try_cast(IntegerType()))
    .withColumn("launch_date", F.to_date(F.col("launch_date"), "yyyy-MM-dd"))
    .withColumn("product_rating", F.col("product_rating").try_cast(DecimalType(3, 2)))
)

In [0]:
# sales_orders
sales_orders = (
    sales_orders
    .withColumn("order_timestamp", F.to_timestamp(F.col("order_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("quantity", F.col("quantity").try_cast(IntegerType()))
    .withColumn("discount_pct", F.col("discount_pct").try_cast(DecimalType(5, 2)))
    .withColumn("promised_delivery_date", F.to_date(F.col("promised_delivery_date"), "yyyy-MM-dd"))
    .withColumn("actual_delivery_date", F.to_date(F.col("actual_delivery_date"), "yyyy-MM-dd"))
)

### Step 5 & 6

In [0]:
# steps 5 & 6 for customers
w = Window.partitionBy("customer_id").orderBy(F.desc_nulls_last("updated_at"))

deduped_customers = (
    customers
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .filter(F.col("is_active") == F.lit("Y"))
)

### Step 7 & 8

In [0]:
products = (
    products
    .filter(
        (F.col("unit_price") > 0) &
        (F.col("cost_price") > 0)
    )
    .filter(F.col("active_flag") == F.lit("Y"))
)

### Step 9 & 10

In [0]:
sales_orders = (
    sales_orders
    .filter(F.col("quantity") > 0)
    .filter(~(F.col("order_status").isin(["CANCELLED", "PENDING"])))
)

### Step 11 & 12

In [0]:
# add delivery_days, late_delivery_flag, and order_month before joins
sales_orders = (
    sales_orders
    .withColumn("order_month", F.date_format(F.col("order_timestamp"), "yyyy-MM"))
    .withColumn(
        "late_delivery_flag",
        F.when(F.col("actual_delivery_date").isNull(), None)
        .when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), F.lit("Y"))
        .otherwise(F.lit("N"))
    )
    .withColumn("delivery_days", F.datediff(F.col("actual_delivery_date"), F.col("order_timestamp")))
)

In [0]:
# join tables for remaining columns
full_table = (
    sales_orders.join(deduped_customers, "customer_id", "inner")
    .join(products, "product_id", "inner")
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
    .withColumn("discount_amount", F.col("gross_amount") * F.coalesce(F.col("discount_pct"), F.lit(0)) / 100)
    .withColumn("net_amount", F.col("gross_amount") - F.col("discount_amount"))
    .withColumn("net_sales", F.when(F.col("order_status") == "COMPLETED", F.col("net_amount")).otherwise(F.lit(0)))
    .withColumn("profit_per_unit", (F.col("net_amount") / F.col("quantity")) - F.col("cost_price"))
    .drop("ingestion_timestamp", "source_file")
)

### Step 13

In [0]:
# save managed tables
customers.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_customers")
products.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_products")
sales_orders.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_sales_orders")
full_table.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_full_table")

### Step 14

In [0]:
display(full_table.limit(5))

In [0]:
# monthly category sales
monthly_cat_sales = (
    full_table
    .select("order_month", "category", "net_sales")
    .groupBy("order_month", "category")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy("order_month", F.col("total_sales").desc())
)
monthly_cat_sales.show(truncate = False)

In [0]:
# city sales
city_sales = (
    full_table
    .select("city", "net_sales")
    .groupBy("city")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

city_sales.show(truncate = False)

In [0]:
# customer value
customer_value = (
    full_table
    .select("customer_id", "customer_name", "net_sales")
    .groupBy("customer_id", "customer_name")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

customer_value.show(truncate = False)

In [0]:
# top products by category
top_products = (
    full_table
    .select("product_id", "product_name", "category", "net_sales")
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("category"), F.col("total_sales").desc())
)

top_products.show(truncate = False)

In [0]:
# save gold tables
monthly_cat_sales.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_monthly_cat_sales")
city_sales.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_city_sales")
customer_value.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_customer_value")
top_products.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_top_products")

## Part C - Aggregation & Window Functions

### Step 1

### Step 2

### Step 3

In [0]:
result_3 = (
    full_table.crossJoin(full_table.agg(F.max("signup_date").alias("max_signup_date")))
    .filter(F.col("signup_date") == F.col("max_signup_date"))
    .select(
        "customer_id", "customer_name", "email", "city", "state", "region",
        "customer_segment", "signup_date", "date_of_birth", "is_active",
        "loyalty_points", "updated_at",
    )
)
result_3.show(n = 10, truncate = False)

### Step 4

In [0]:
result_4 = (
    full_table.groupBy("category", "product_name")
    .agg(F.round(F.sum("net_sales"), 2).alias("total_net_sales"))
    .withColumn(
        "product_rank",
        F.rank().over(
            Window.partitionBy("category").orderBy(F.desc("total_net_sales"))
        ),
    )
    .orderBy("category", "product_rank")
)
result_4.show(n = 10, truncate = False)

### Step 5

In [0]:
result_5 = (
    full_table.groupBy("state", "customer_id", "customer_name")
    .agg(F.sum("net_amount").alias("total_net_amount"))
    .withColumn(
        "rank",
        F.dense_rank().over(
            Window.partitionBy("state").orderBy(F.desc("total_net_amount"))
        ),
    )
    .select("state", "customer_id", "customer_name", "rank")
    .orderBy("state", "rank")
)
result_5.show(n = 10, truncate = False)

### Step 6

In [0]:
monthly = (
    full_table.groupBy("category", "order_month")
    .agg(F.round(F.sum("net_sales"), 2).alias("monthly_revenue"))
)
running_w = (
    Window.partitionBy("category")
    .orderBy("order_month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
result_6 = (
    monthly
    .withColumn("running_revenue", F.round(F.sum("monthly_revenue").over(running_w), 2))
    .orderBy("category", "order_month")
)
result_6.show(n = 10, truncate = False)

### Step 7

In [0]:
w = Window.partitionBy("customer_id").orderBy(F.desc("order_timestamp"))
result_7 = (
    full_table.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("customer_id")
)
result_7.show(n = 10, truncate = False)

## Part D - Spark SQL

### Part 1

In [0]:
%sql
SELECT *
FROM workspace.retail_fresher.silver_sales_orders
WHERE order_status = 'COMPLETED'

### Part 2

In [0]:
%sql
SELECT *
FROM workspace.retail_fresher.silver_sales_orders o
JOIN workspace.retail_fresher.silver_products p ON o.product_id = p.product_id
JOIN workspace.retail_fresher.silver_customers c ON o.customer_id = c.customer_id 

### Part 3

### Part 4

In [0]:
%sql
SELECT
  category,
  order_month AS sales_month,
  COUNT(DISTINCT order_id) AS order_count,
  SUM(quantity) AS total_quantity,
  ROUND(SUM(net_sales), 2) AS total_revenue,
  ROUND(AVG(net_sales), 2) AS avg_net_sale,
  ROUND(MIN(net_sales), 2) AS min_net_sale,
  ROUND(MAX(net_sales), 2) AS max_net_sale
FROM workspace.retail_fresher.silver_full_table
WHERE order_status = 'COMPLETED'
GROUP BY category, order_month
ORDER BY category, sales_month;

### Part 5

In [0]:
%sql
SELECT
  category,
  product_id,
  product_name,
  ROUND(SUM(net_sales), 2) AS category_revenue,
  RANK() OVER (
    PARTITION BY category
    ORDER BY SUM(net_sales) DESC
  ) AS product_rank
FROM workspace.retail_fresher.silver_full_table
WHERE order_status = 'COMPLETED'
GROUP BY category, product_id, product_name
ORDER BY category, product_rank;

### Part 6

In [0]:
%sql
WITH customer_sales AS (
  SELECT
    state,
    customer_id,
    customer_name,
    ROUND(SUM(net_sales), 2) AS total_revenue
  FROM workspace.retail_fresher.silver_full_table
  WHERE order_status = 'COMPLETED'
  GROUP BY state, customer_id, customer_name
),
ranked AS (
  SELECT
    state,
    customer_id,
    customer_name,
    total_revenue,
    DENSE_RANK() OVER (
      PARTITION BY state
      ORDER BY total_revenue DESC
    ) AS customer_rank
  FROM customer_sales
)
SELECT *
FROM ranked
WHERE customer_rank <= 3
ORDER BY state, customer_rank;

### Part 7